In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Explainable Multi-Entity Fraud Intelligence Framework

## JP Morgan Data Analytics Internship Project

In [2]:
transactions = pd.read_csv('../../B_Dataset/raw/transactions.csv')

customers = pd.read_csv('../../B_Dataset/raw/customers.csv')

merchants = pd.read_csv('../../B_Dataset/raw/merchants.csv')

In [3]:
print("Transactions Shape:", transactions.shape)

print("Customers Shape:", customers.shape)

print("Merchants Shape:", merchants.shape)

Transactions Shape: (5000, 17)
Customers Shape: (5000, 8)
Merchants Shape: (5000, 4)


In [4]:
transactions.columns
customers.columns
merchants.columns

Index(['merchant_id', 'merchant_category', 'merchant_risk_score',
       'channel_default'],
      dtype='str')

In [5]:
transactions.columns

Index(['transaction_id', 'event_ts', 'customer_id', 'merchant_id', 'channel',
       'transaction_amount_usd', 'txn_country', 'txn_hour',
       'device_risk_score', 'new_device_flag', 'velocity_1h', 'velocity_24h',
       'geo_distance_km', 'merchant_risk_score', 'is_night_flag',
       'alert_generated', 'fraud_label'],
      dtype='str')

In [6]:
customers.columns

Index(['customer_id', 'age', 'tenure_months', 'segment', 'home_country',
       'digital_only', 'kyc_risk_band', 'synthetic_identity_score'],
      dtype='str')

In [7]:
transactions.merge(customers, on="customer_id")
transactions.merge(merchants, on="merchant_id")

,transaction_id,event_ts,customer_id,merchant_id,channel,transaction_amount_usd,txn_country,txn_hour,device_risk_score,new_device_flag,velocity_1h,velocity_24h,geo_distance_km,merchant_risk_score_x,is_night_flag,alert_generated,fraud_label,merchant_category,merchant_risk_score_y,channel_default
0,T9028450,2025-01-12 06:38:00,C100000,M100000,Card Present,70.46,US,4,0.747,1,1,2,3.0,0.474,1,0,0,Utilities,0.554,Card Not Present
1,T9050670,2025-01-20 04:18:00,C100001,M100001,P2P,36.12,DE,3,0.197,0,1,3,1488.6,0.218,1,1,0,Digital Wallet,0.162,P2P
2,T9015811,2025-03-16 21:19:00,C100002,M100002,Card Not Present,156.38,SG,10,0.393,0,3,6,136.2,0.632,0,0,0,Travel,0.449,Card Not Present
3,T9014668,2025-02-24 19:35:00,C100003,M100003,Card Present,97.55,US,6,0.125,0,3,5,30.5,0.632,0,0,0,Electronics,0.315,Online Banking
4,T9057899,2025-01-30 14:22:00,C100004,M100004,Online Banking,160.49,AE,12,0.138,0,1,3,3.1,0.097,0,0,0,Telecom,0.079,Online Banking
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,T9002463,2025-03-29 08:35:00,C104995,M104995,Card Present,638.74,IN,13,0.576,0,0,3,4.1,0.097,0,0,0,Grocery,0.094,P2P
4996,T9025576,2025-02-17 02:31:00,C104996,M104996,Card Not Present,27.04,US,10,0.690,0,1,4,33.8,0.111,0,0,0,Marketplace,0.502,Wire
4997,T9000776,2025-03-06 23:26:00,C104997,M104997,P2P,122.38,US,13,0.345,0,0,2,3.6,0.450,0,0,0,Utilities,0.721,Online Banking
4998,T9012418,2025-04-27 15:29:00,C104998,M104998,Online Banking,24.51,US,2,0.412,0,1,3,6.4,0.374,1,0,0,Gaming,0.100,Wire


In [8]:
df = transactions \
    .merge(customers, on="customer_id", how="left") \
    .merge(merchants, on="merchant_id", how="left")

In [9]:
df["transaction_hour"] = pd.to_datetime(df["timestamp"]).dt.hour
df["is_weekend"] = pd.to_datetime(df["timestamp"]).dt.dayofweek >= 5

KeyError: 'timestamp'

In [10]:
transactions.columns

Index(['transaction_id', 'event_ts', 'customer_id', 'merchant_id', 'channel',
       'transaction_amount_usd', 'txn_country', 'txn_hour',
       'device_risk_score', 'new_device_flag', 'velocity_1h', 'velocity_24h',
       'geo_distance_km', 'merchant_risk_score', 'is_night_flag',
       'alert_generated', 'fraud_label'],
      dtype='str')

In [11]:
df["event_ts"] = pd.to_datetime(df["event_ts"])

In [12]:
df["hour"] = df["event_ts"].dt.hour
df["dayofweek"] = df["event_ts"].dt.dayofweek
df["is_weekend"] = df["dayofweek"] >= 5

In [13]:
df["combined_risk"] = (
    df["device_risk_score"] +
    df["merchant_risk_score"]
)

KeyError: 'merchant_risk_score'

In [14]:
df.columns

Index(['transaction_id', 'event_ts', 'customer_id', 'merchant_id', 'channel',
       'transaction_amount_usd', 'txn_country', 'txn_hour',
       'device_risk_score', 'new_device_flag', 'velocity_1h', 'velocity_24h',
       'geo_distance_km', 'merchant_risk_score_x', 'is_night_flag',
       'alert_generated', 'fraud_label', 'age', 'tenure_months', 'segment',
       'home_country', 'digital_only', 'kyc_risk_band',
       'synthetic_identity_score', 'merchant_category',
       'merchant_risk_score_y', 'channel_default', 'hour', 'dayofweek',
       'is_weekend'],
      dtype='str')

In [15]:
[col for col in df.columns if "risk" in col.lower()]

['device_risk_score',
 'merchant_risk_score_x',
 'kyc_risk_band',
 'merchant_risk_score_y']

In [16]:
df["merchant_risk_score"] = df[["merchant_risk_score_x", "merchant_risk_score_y"]].mean(axis=1)
df.drop(columns=["merchant_risk_score_x", "merchant_risk_score_y"], inplace=True)

In [17]:
df["kyc_risk_band"].value_counts()

kyc_risk_band
Low       2669
Medium    1179
High       330
Name: count, dtype: int64

In [18]:
df = pd.get_dummies(df, columns=["kyc_risk_band"], drop_first=True)

In [19]:
df["total_risk_score"] = (
    df["device_risk_score"] +
    df["merchant_risk_score"]
)

In [20]:
df[["device_risk_score", "merchant_risk_score_x", "merchant_risk_score_y"]].head()

KeyError: "['merchant_risk_score_x', 'merchant_risk_score_y'] not in index"

In [21]:
[col for col in df.columns if "risk" in col.lower()]

['device_risk_score',
 'merchant_risk_score',
 'kyc_risk_band_Low',
 'kyc_risk_band_Medium',
 'total_risk_score']

In [22]:
df.columns.tolist()

['transaction_id',
 'event_ts',
 'customer_id',
 'merchant_id',
 'channel',
 'transaction_amount_usd',
 'txn_country',
 'txn_hour',
 'device_risk_score',
 'new_device_flag',
 'velocity_1h',
 'velocity_24h',
 'geo_distance_km',
 'is_night_flag',
 'alert_generated',
 'fraud_label',
 'age',
 'tenure_months',
 'segment',
 'home_country',
 'digital_only',
 'synthetic_identity_score',
 'merchant_category',
 'channel_default',
 'hour',
 'dayofweek',
 'is_weekend',
 'merchant_risk_score',
 'kyc_risk_band_Low',
 'kyc_risk_band_Medium',
 'total_risk_score']

In [23]:
risk_cols = [c for c in df.columns if "risk" in c.lower()]
risk_cols

['device_risk_score',
 'merchant_risk_score',
 'kyc_risk_band_Low',
 'kyc_risk_band_Medium',
 'total_risk_score']

In [24]:
df["total_risk_score"] = df[risk_cols].mean(axis=1)

In [25]:
"merchant_risk_score" in df.columns

True

In [26]:
df[["device_risk_score", "merchant_risk_score"]].head()

,device_risk_score,merchant_risk_score
0,0.747,0.5140
1,0.197,0.1900
2,0.393,0.5405
3,0.125,0.4735
4,0.138,0.0880


In [27]:
df["total_risk_score"] = (
    df["device_risk_score"] +
    df["merchant_risk_score"]
)

In [28]:
df["risk_gap"] = abs(df["device_risk_score"] - df["merchant_risk_score"])

In [ ]:
df["fraud_label"].value_counts(normalize=True)

In [29]:
df.groupby("alert_generated")["fraud_label"].mean()

alert_generated
0    0.032716
1    0.300000
Name: fraud_label, dtype: float64

In [30]:
X = df.drop(columns=["fraud_label", "transaction_id", "event_ts"])
y = df["fraud_label"]

In [31]:
X = df.drop(columns=["fraud_label", "transaction_id", "event_ts"])
y = df["fraud_label"]

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

ModuleNotFoundError: No module named 'sklearn'

In [33]:
pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl (37.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ------------------

In [34]:
import sklearn
print(sklearn.__version__)

1.8.0


In [35]:
from sklearn.model_selection import train_test_split

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

ValueError: could not convert string to float: 'C104496'

In [38]:
X.select_dtypes(include="object").columns

C:\Users\karthik\AppData\Local\Temp\ipykernel_27948\2213039483.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include="object").columns


Index(['customer_id', 'merchant_id', 'channel', 'txn_country', 'tenure_months',
       'segment', 'home_country', 'merchant_category', 'channel_default'],
      dtype='str')

In [39]:
id_cols = ["customer_id", "merchant_id"]
X = X.drop(columns=id_cols)

In [40]:
cat_cols = [
    "channel",
    "txn_country",
    "segment",
    "home_country",
    "merchant_category",
    "channel_default"
]

In [41]:
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

In [42]:
df["tenure_months"].dtype

<StringDtype(storage='python', na_value=nan)>

In [43]:
X["tenure_months"] = pd.to_numeric(X["tenure_months"], errors="coerce")

In [44]:
# 1. Drop IDs
X = X.drop(columns=["customer_id", "merchant_id"])

# 2. Fix numeric column if needed
X["tenure_months"] = pd.to_numeric(X["tenure_months"], errors="coerce")

# 3. One-hot encode categoricals
X = pd.get_dummies(X, columns=[
    "channel",
    "txn_country",
    "segment",
    "home_country",
    "merchant_category",
    "channel_default"
], drop_first=True)

KeyError: "['customer_id', 'merchant_id'] not found in axis"

In [45]:
X.columns.tolist()

['transaction_amount_usd',
 'txn_hour',
 'device_risk_score',
 'new_device_flag',
 'velocity_1h',
 'velocity_24h',
 'geo_distance_km',
 'is_night_flag',
 'alert_generated',
 'age',
 'tenure_months',
 'digital_only',
 'synthetic_identity_score',
 'hour',
 'dayofweek',
 'is_weekend',
 'merchant_risk_score',
 'kyc_risk_band_Low',
 'kyc_risk_band_Medium',
 'total_risk_score',
 'risk_gap',
 'channel_Card Present',
 'channel_Online Banking',
 'channel_P2P',
 'channel_Wire',
 'txn_country_BR',
 'txn_country_CA',
 'txn_country_DE',
 'txn_country_GB',
 'txn_country_IN',
 'txn_country_NG',
 'txn_country_SG',
 'txn_country_UK',
 'txn_country_US',
 'segment_Mass',
 'segment_SME',
 'home_country_CA',
 'home_country_IN',
 'home_country_SG',
 'home_country_UK',
 'home_country_US',
 'merchant_category_Electronics',
 'merchant_category_Fuel',
 'merchant_category_Gaming',
 'merchant_category_Grocery',
 'merchant_category_Luxury',
 'merchant_category_Marketplace',
 'merchant_category_Telecom',
 'merchant

In [46]:
X = df.drop(columns=["fraud_label", "transaction_id", "event_ts"], errors="ignore")
y = df["fraud_label"]

In [47]:
# Convert numeric safely
X["tenure_months"] = pd.to_numeric(X["tenure_months"], errors="coerce")

# Identify categorical columns
cat_cols = X.select_dtypes(include="object").columns

# One-hot encode all categoricals
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

C:\Users\karthik\AppData\Local\Temp\ipykernel_27948\926297087.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [48]:
df → X, y → clean → encode → train

SyntaxError: invalid character '→' (U+2192) (662866405.py, line 1)

In [49]:
from sklearn.model_selection import train_test_split

In [50]:
X.columns.tolist()

['transaction_amount_usd',
 'txn_hour',
 'device_risk_score',
 'new_device_flag',
 'velocity_1h',
 'velocity_24h',
 'geo_distance_km',
 'is_night_flag',
 'alert_generated',
 'age',
 'tenure_months',
 'digital_only',
 'synthetic_identity_score',
 'hour',
 'dayofweek',
 'is_weekend',
 'merchant_risk_score',
 'kyc_risk_band_Low',
 'kyc_risk_band_Medium',
 'total_risk_score',
 'risk_gap',
 'customer_id_C100001',
 'customer_id_C100002',
 'customer_id_C100003',
 'customer_id_C100004',
 'customer_id_C100005',
 'customer_id_C100006',
 'customer_id_C100007',
 'customer_id_C100008',
 'customer_id_C100009',
 'customer_id_C100010',
 'customer_id_C100011',
 'customer_id_C100012',
 'customer_id_C100013',
 'customer_id_C100014',
 'customer_id_C100015',
 'customer_id_C100016',
 'customer_id_C100017',
 'customer_id_C100018',
 'customer_id_C100019',
 'customer_id_C100020',
 'customer_id_C100021',
 'customer_id_C100022',
 'customer_id_C100023',
 'customer_id_C100024',
 'customer_id_C100025',
 'customer_i

In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [1]:
# Data handling
import pandas as pd
import numpy as np

# Train-test split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.impute import SimpleImputer

# Model
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Evaluation
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [52]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [53]:
X.isna().sum().sort_values(ascending=False).head(20)

tenure_months                     5000
age                                822
digital_only                       822
synthetic_identity_score           822
device_risk_score                    0
home_country_UK                      0
home_country_US                      0
merchant_category_Electronics        0
merchant_category_Fuel               0
merchant_category_Gaming             0
merchant_category_Grocery            0
merchant_category_Luxury             0
merchant_category_Marketplace        0
merchant_category_Telecom            0
merchant_category_Travel             0
merchant_category_Utilities          0
channel_default_Card Present         0
channel_default_Online Banking       0
channel_default_P2P                  0
channel_default_Wire                 0
dtype: int64

In [54]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

c:\Users\karthik\OneDrive\Desktop\NextLeap\JP Morgan Internship\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['tenure_months']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\karthik\OneDrive\Desktop\NextLeap\JP Morgan Internship\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['tenure_months']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [55]:
X = X.drop(columns=["tenure_months"])

In [56]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [57]:
pipeline.fit(X_train, y_train)

NameError: name 'pipeline' is not defined

In [58]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

In [59]:
pipeline.fit(X_train, y_train)

c:\Users\karthik\OneDrive\Desktop\NextLeap\JP Morgan Internship\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

In [2]:
# Data handling
import pandas as pd
import numpy as np

# Train-test split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.impute import SimpleImputer

# Model
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Evaluation
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [3]:
# Separate features and target variable
# Dropping identifiers and timestamp since they do not help prediction
X = df.drop(columns=["fraud_label", "transaction_id", "event_ts"], errors="ignore")
y = df["fraud_label"]

NameError: name 'df' is not defined

In [4]:
# Identify categorical columns (text/string columns)
cat_cols = X.select_dtypes(include="object").columns

# Convert categorical columns into numeric format
# This is required because Logistic Regression only works with numbers
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

NameError: name 'X' is not defined

In [5]:
# Separate features and target variable
# Dropping identifiers and timestamp since they do not help prediction

X = df.drop(columns=["fraud_label", "transaction_id", "event_ts"], errors="ignore")
y = df["fraud_label"]

NameError: name 'df' is not defined

In [6]:
transactions.head()


NameError: name 'transactions' is not defined

In [7]:
transactions.head()

NameError: name 'transactions' is not defined

In [8]:
import pandas as pd

In [9]:
# Import pandas library
# Pandas is used for loading, cleaning, and analyzing datasets

import pandas as pd

In [10]:
# Import os library
# os helps interact with files and folders in the current working directory

import os

# Display all files available in the current notebook folder
# This helps identify dataset filenames before loading them

print(os.listdir())

['jp_morgan_analysis.ipynb', 'jp_morgan_fraud_intelligence.ipynb', 'starter_notebook.ipynb']


In [11]:
['transactions.csv', 'customers.csv', 'merchants.csv']

['transactions.csv', 'customers.csv', 'merchants.csv']

In [12]:
# Load transaction dataset
# Contains transaction-level information such as amount, risk score, velocity, etc.

transactions = pd.read_csv("transactions.csv")

# Load customer dataset
# Contains customer-related attributes such as segment, tenure, country, etc.

customers = pd.read_csv("customers.csv")

# Load merchant dataset
# Contains merchant-related information such as merchant category and risk score

merchants = pd.read_csv("merchants.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'transactions.csv'

In [13]:
# Display all folders and files in current working directory
# Helps identify where datasets are actually stored

import os

for root, dirs, files in os.walk("."):
    print("FOLDER:", root)
    print("FILES:", files)
    print("-" * 40)

FOLDER: .
FILES: ['jp_morgan_analysis.ipynb', 'jp_morgan_fraud_intelligence.ipynb', 'starter_notebook.ipynb']
----------------------------------------


In [14]:
# Display all folders and files in current working directory
# Helps identify where datasets are actually stored

import os

for root, dirs, files in os.walk("."):
    print("FOLDER:", root)
    print("FILES:", files)
    print("-" * 40)

FOLDER: .
FILES: ['jp_morgan_analysis.ipynb', 'jp_morgan_fraud_intelligence.ipynb', 'starter_notebook.ipynb']
----------------------------------------


In [15]:
import os
print(os.listdir())

['customers.csv', 'jp_morgan_analysis.ipynb', 'jp_morgan_fraud_intelligence.ipynb', 'merchants.csv', 'starter_notebook.ipynb', 'transactions.csv']


In [16]:
# Load transaction dataset
# Contains transaction-level fraud activity and behavioral features

transactions = pd.read_csv("transactions.csv")

# Load customer dataset
# Contains customer profile and demographic attributes

customers = pd.read_csv("customers.csv")

# Load merchant dataset
# Contains merchant risk and category information

merchants = pd.read_csv("merchants.csv")

In [17]:
# Display first 5 rows to confirm successful loading

transactions.head()

,transaction_id,event_ts,customer_id,merchant_id,channel,transaction_amount_usd,txn_country,txn_hour,device_risk_score,new_device_flag,velocity_1h,velocity_24h,geo_distance_km,merchant_risk_score,is_night_flag,alert_generated,fraud_label
0,T9028450,2025-01-12 06:38:00,C100000,M100000,Card Present,70.46,US,4,0.747,1,1,2,3.0,0.474,1,0,0
1,T9050670,2025-01-20 04:18:00,C100001,M100001,P2P,36.12,DE,3,0.197,0,1,3,1488.6,0.218,1,1,0
2,T9015811,2025-03-16 21:19:00,C100002,M100002,Card Not Present,156.38,SG,10,0.393,0,3,6,136.2,0.632,0,0,0
3,T9014668,2025-02-24 19:35:00,C100003,M100003,Card Present,97.55,US,6,0.125,0,3,5,30.5,0.632,0,0,0
4,T9057899,2025-01-30 14:22:00,C100004,M100004,Online Banking,160.49,AE,12,0.138,0,1,3,3.1,0.097,0,0,0


In [18]:
print(os.listdir())

['customers.csv', 'jp_morgan_analysis.ipynb', 'jp_morgan_fraud_intelligence.ipynb', 'merchants.csv', 'starter_notebook.ipynb', 'transactions.csv']


In [19]:
# Create a working copy of transaction dataset
# Using .copy() prevents accidental modification of original raw data

df = transactions.copy()

In [20]:
# Separate features (X) and target variable (y)

# fraud_label is the target variable we want to predict
# transaction_id and event_ts are removed because:
# - transaction_id is only a unique identifier
# - raw timestamps are not directly useful for Logistic Regression

X = df.drop(columns=["fraud_label", "transaction_id", "event_ts"], errors="ignore")

# Target variable:
# 0 = legitimate transaction
# 1 = fraudulent transaction

y = df["fraud_label"]

In [21]:
# Identify categorical/text columns
# Machine learning models require numeric input

cat_cols = X.select_dtypes(include="object").columns

# Convert categorical variables into numeric format using one-hot encoding
# drop_first=True prevents duplicate information

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

C:\Users\karthik\AppData\Local\Temp\ipykernel_7932\4048616332.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [22]:
# Display final dataset dimensions after encoding

print("Feature Matrix Shape:", X.shape)
print("Target Variable Shape:", y.shape)

Feature Matrix Shape: (5000, 10021)
Target Variable Shape: (5000,)


In [23]:
X.shape

(5000, 10021)

In [24]:
# Recreate feature matrix from original dataframe
# This ensures we start from clean data before encoding

X = df.drop(
    columns=[
        "fraud_label",
        "transaction_id",
        "event_ts",
        "customer_id",
        "merchant_id"
    ],
    errors="ignore"
)

# Define target variable
y = df["fraud_label"]

In [25]:
# Identify categorical/text columns
cat_cols = X.select_dtypes(include="object").columns

# Apply one-hot encoding to categorical variables
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

C:\Users\karthik\AppData\Local\Temp\ipykernel_7932\800341660.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [26]:
# Display final dataset dimensions after proper encoding

print("Feature Matrix Shape:", X.shape)

Feature Matrix Shape: (5000, 23)


In [27]:
# Import train_test_split function
# Used to divide dataset into training and testing subsets

from sklearn.model_selection import train_test_split

# Split data into:
# - Training set (80%) → used for learning patterns
# - Test set (20%) → used for evaluating performance on unseen data

# stratify=y ensures fraud ratio remains consistent in both datasets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
# Display dimensions of train and test datasets

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("Training Labels Shape:", y_train.shape)
print("Testing Labels Shape:", y_test.shape)

Training Features Shape: (4000, 23)
Testing Features Shape: (1000, 23)
Training Labels Shape: (4000,)
Testing Labels Shape: (1000,)


In [29]:
# Import preprocessing and modeling tools

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [30]:
# Build machine learning pipeline

# Pipeline performs:
# 1. Missing value handling using median imputation
# 2. Logistic Regression model training

# Median is used because fraud datasets often contain outliers

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [31]:
# Train the fraud detection model

# The model learns relationships between:
# - transaction behavior
# - customer/merchant risk
# - fraud outcomes

pipeline.fit(X_train, y_train)

c:\Users\karthik\OneDrive\Desktop\NextLeap\JP Morgan Internship\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

In [32]:
# Import evaluation metrics

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [33]:
# Predict fraud labels on unseen test data

y_pred = pipeline.predict(X_test)

# Predict fraud probabilities
# Needed for ROC-AUC evaluation

y_prob = pipeline.predict_proba(X_test)[:, 1]

In [34]:
# Evaluate fraud detection performance

# ROC-AUC measures how well the model separates fraud vs non-fraud transactions

print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

# Confusion matrix shows:
# - True Positives
# - False Positives
# - False Negatives
# - True Negatives

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification report provides:
# - Precision
# - Recall
# - F1-score

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

ROC-AUC Score: 0.644296875

Confusion Matrix:
[[743 217]
 [ 20  20]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.77      0.86       960
           1       0.08      0.50      0.14        40

    accuracy                           0.76      1000
   macro avg       0.53      0.64      0.50      1000
weighted avg       0.94      0.76      0.83      1000



## Model Evaluation and Interpretation

The Logistic Regression model achieved an ROC-AUC score of 0.64, indicating moderate ability to distinguish between fraudulent and legitimate transactions.

The model achieved a recall of 0.50 for the fraud class, meaning that 50% of fraudulent transactions were successfully identified.

Recall is prioritized in fraud detection because missing fraudulent activity is more costly than generating additional alerts.

However, precision for fraud detection was relatively low (0.08), indicating a high number of false positive alerts. This suggests that while the model captures fraud cases reasonably well, further optimization is required to reduce unnecessary fraud alerts.

Overall, the Logistic Regression model serves as a useful baseline model for fraud risk detection and demonstrates the complete machine learning workflow from preprocessing to evaluation.


In [35]:
# Apply custom fraud probability threshold

# Lowering threshold increases fraud sensitivity
# This helps capture more fraudulent transactions

custom_threshold = 0.30

# Convert probabilities into fraud predictions
# If probability >= threshold → predict fraud (1)

y_pred_custom = (y_prob >= custom_threshold).astype(int)

In [36]:
# Evaluate model performance using custom threshold

print("Custom Threshold:", custom_threshold)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_custom))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_custom))

Custom Threshold: 0.3

Confusion Matrix:
[[197 763]
 [  8  32]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.21      0.34       960
           1       0.04      0.80      0.08        40

    accuracy                           0.23      1000
   macro avg       0.50      0.50      0.21      1000
weighted avg       0.92      0.23      0.33      1000



## Threshold Tuning Results

To improve fraud detection recall, the classification threshold was lowered from the default value of 0.50 to 0.30.

This adjustment increased fraud recall from 50% to 80%, meaning the model successfully detected a larger proportion of fraudulent transactions.

However, the lower threshold also significantly increased false positive alerts, reducing precision. This highlights the common tradeoff in fraud detection systems between maximizing fraud capture and minimizing operational alert volume.

Such threshold tuning reflects real-world banking practices, where thresholds are adjusted based on business risk tolerance and fraud investigation capacity.

In [37]:
# Check whether XGBoost is installed

import xgboost
print("XGBoost installed successfully")

ModuleNotFoundError: No module named 'xgboost'

In [38]:
# Install XGBoost library
# XGBoost is a powerful gradient boosting algorithm widely used in fraud detection

!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   -- ------------------------------------- 6.6/101.7 MB 27.8 MB/s eta 0:00:04
   ---- ----------------------------------- 12.1/101.7 MB 26.6 MB/s eta 0:00:04
   ------- -------------------------------- 18.9/101.7 MB 28.5 MB/s eta 0:00:03
   ---------- ----------------------------- 26.0/101.7 MB 29.4 MB/s eta 0:00:03
   ------------ --------------------------- 32.8/101.7 MB 29.9 MB/s eta 0:00:03
   --------------- ------------------------ 39.6/101.7 MB 30.3 MB/s eta 0:00:03
   ----------------- ---------------------- 45.6/101.7 MB 30.0 MB/s eta 0:00:02
   -------------------- ------------------- 53.0/101.7 MB 30.3 MB/s eta 0:00:02
   ----------------------- ---------------- 59.5/101.7 MB 30.4 MB/s eta 0:00:02
   -------------------------- ------------- 66.6/101.7 MB 30.6 MB/s eta 0:00:02
   ---------------------------- ----------- 73.4/101.7 MB 3

In [39]:
# Verify XGBoost installation

import xgboost

print("XGBoost installed successfully")

XGBoost installed successfully


In [40]:
# Verify XGBoost installation

import xgboost

print("XGBoost installed successfully")

XGBoost installed successfully


In [42]:
# Import XGBoost classifier
# XGBoost is an advanced gradient boosting algorithm widely used in fraud detection

from xgboost import XGBClassifier

In [41]:
# Create XGBoost fraud detection model

# Key configurations:
# - scale_pos_weight handles fraud imbalance
# - random_state ensures reproducibility
# - eval_metric avoids warning messages

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=24,
    random_state=42,
    eval_metric="logloss"
)

NameError: name 'XGBClassifier' is not defined

In [43]:
# Import XGBoost classifier
# XGBoost is an advanced gradient boosting algorithm widely used in fraud detection

from xgboost import XGBClassifier

In [44]:
# Create XGBoost fraud detection model

# Key configurations:
# - n_estimators controls number of boosting trees
# - max_depth controls tree complexity
# - learning_rate controls how quickly the model learns
# - scale_pos_weight helps handle fraud class imbalance
# - random_state ensures reproducible results
# - eval_metric avoids default warning messages

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=24,
    random_state=42,
    eval_metric="logloss"
)

In [45]:
# Train XGBoost fraud detection model

xgb_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [46]:
# Generate fraud predictions using XGBoost model

# Predict fraud labels on unseen test data

xgb_pred = xgb_model.predict(X_test)

# Predict fraud probabilities
# These probabilities are used for ROC-AUC evaluation

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

In [47]:
# Evaluate XGBoost fraud detection performance

print("XGBoost ROC-AUC Score:", roc_auc_score(y_test, xgb_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

print("\nClassification Report:")
print(classification_report(y_test, xgb_pred))

XGBoost ROC-AUC Score: 0.56265625

Confusion Matrix:
[[873  87]
 [ 30  10]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94       960
           1       0.10      0.25      0.15        40

    accuracy                           0.88      1000
   macro avg       0.53      0.58      0.54      1000
weighted avg       0.93      0.88      0.91      1000



## Final Model Comparison

Two machine learning models were evaluated for fraud detection:

1. Logistic Regression
2. XGBoost Classifier

The Logistic Regression model achieved a higher ROC-AUC score and significantly better fraud recall, detecting a larger proportion of fraudulent transactions.

The XGBoost model reduced false positive alerts but detected fewer fraud cases overall.

This comparison demonstrates that more complex models do not always outperform simpler baseline models, particularly when working with smaller datasets and limited fraud samples.

For this dataset, Logistic Regression provided better fraud detection sensitivity, while XGBoost offered improved alert precision and operational efficiency.

The project highlights the importance of evaluating machine learning models based on business objectives rather than relying solely on model complexity.

## Tools and Technologies Used

- Python
- pandas
- scikit-learn
- XGBoost
- matplotlib
- Jupyter Notebook / VS Code

The analysis workflow was implemented entirely within Python using pandas-based data processing and machine learning libraries.